# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 13** you build `fp.Design` **binder** objects from the pool, filtering on the
**engaged** signaling subunits (β, γc), call `fp.run_pipeline(..., design_type="binder")`, and
`fp.report(...)` the survival funnel + ranked CSV (D3 part 1). The **selectivity** modeling (engages βγ
but not α) is layered on top in notebook 04.

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/rfdiffusion_designs.csv` (+ `bindcraft_designs.csv`) exist.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6. We apply `pae` to
the **worst engaged subunit** (β or γc) — a design must confidently engage **both** signaling chains.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

## Build `Design` (binder) objects from the pool

Map each pool row onto `fp.Design` with `design_type="binder"`. The binder metrics drive the layers:
`scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency — here `pae_interaction` is the **worst
engaged-subunit** pae, i.e. `max(pae_to_beta, pae_to_gamma)`), and
`rosetta_dG`/`shape_complementarity`/`solubility` (Layer 3 physics). We keep the **per-subunit** paes +
`paradigm` in `extra` for the selectivity analysis in notebook 04. (Mock has no independent orthogonal
predictor, so we run Layers 1+3 here; on Colab add a second predictor for Layer 2.)

In [ ]:
import os
import pandas as pd

# Regenerate the pool if a fresh session lost it (deterministic mock).
if not os.path.exists("results/rfdiffusion_designs.csv"):
    import cytokine_tools as ct
    TARGET, HOTSPOTS = "IL2R_beta_gamma", ct.parse_hotspots("B41,B42,C100,C102")
    rf = ct.generate_agonists_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock")
    ct.score_designs(rf, tool="mock")
    rows=[]
    for d in rf:
        pae=d.pae_by_subunit or {}
        rows.append(dict(design_id=d.design_id, paradigm=d.paradigm, length=d.length, sequence=d.sequence,
                         plddt=d.plddt, scrmsd=d.scrmsd, shape_complementarity=d.shape_complementarity,
                         pae_to_alpha=pae.get("IL2Ra"), pae_to_beta=pae.get("IL2Rb"), pae_to_gamma=pae.get("gammaC"),
                         pae_interaction=d.pae_interaction,
                         rosetta_dG=round(-45.0+(ct._hashints("dG",d.design_id)%40),2), solubility=0.3,
                         hotspot_overlap=ct.hotspot_overlap(d.contact_residues,d.hotspots), synthetic=d.synthetic))
    pd.DataFrame(rows).to_csv("results/rfdiffusion_designs.csv", index=False)

# Combine whichever pools exist (RFdiffusion primary; BindCraft foil if present).
frames = [pd.read_csv("results/rfdiffusion_designs.csv")]
if os.path.exists("results/bindcraft_designs.csv"):
    frames.append(pd.read_csv("results/bindcraft_designs.csv"))
pool = pd.concat(frames, ignore_index=True)

def worst_engaged_pae(r):
    vals = [r.get("pae_to_beta"), r.get("pae_to_gamma")]
    vals = [v for v in vals if pd.notna(v)]
    return max(vals) if vals else r.get("pae_interaction")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=worst_engaged_pae(r), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"paradigm": r.get("paradigm"),
               "pae_to_alpha": r.get("pae_to_alpha"), "pae_to_beta": r.get("pae_to_beta"),
               "pae_to_gamma": r.get("pae_to_gamma"), "hotspot_overlap": r.get("hotspot_overlap")},
    )

binders = [row_to_binder(r) for _, r in pool.iterrows()]
print(f"built {len(binders)} binder Designs (pae_interaction = worst engaged subunit, β/γc)")

## Run the pipeline (engaged-subunit cutoffs)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked DataFrame
with survival counts in `df.attrs`. Filtering here is on the **engaged** signaling chains (a design must
confidently engage both β and γc). We use Layers 1+3 (mock has no independent orthogonal source; add
Layer 2 on Colab with a second predictor). **Selectivity** (sparing α) is enforced in notebook 04.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

df_ranked = fp.run_pipeline(binders, design_type="binder", use_layers=(1, 3))
# carry the per-subunit paes + paradigm out of extra for downstream use
for col in ["paradigm", "pae_to_alpha", "pae_to_beta", "pae_to_gamma", "hotspot_overlap"]:
    df_ranked[col] = df_ranked["extra"].apply(lambda e: e.get(col) if isinstance(e, dict) else None)

surv = df_ranked.attrs["survival"]; n = df_ranked.attrs["n_total"]
passed = int((df_ranked["layers_passed"] >= 3).sum())
print(f"pool: {n} designs, survival {surv}, all-layers (engaged) hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")

df_ranked.to_csv("results/all_ranked.csv", index=False)
print("wrote results/all_ranked.csv", df_ranked.shape)
df_ranked.head(10)[["design_id", "paradigm", "layers_passed", "score",
                    "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Read the bars as a funnel:
steep drops show which layer discriminates. (This counts designs that confidently **engage** β/γc; the
selectivity filter in notebook 04 then asks which of these also **spare** α.)

In [ ]:
top = fp.report(df_ranked, top_n=15, save_prefix="results/p13")
print("\nsaved results/p13_survival.png + results/p13_ranked.csv")
top

## Honest hit-rate accounting

Report `N passing all (engaged) layers / N generated`. This is the **engagement** hit rate; notebook 04
adds the **selectivity** hit rate (how many engaged survivors also spare α). Remember: survival is
*enrichment*, not *correctness*, and engaging β/γc is not yet agonism. Mock numbers are SYNTHETIC.

In [ ]:
dist = df_ranked["layers_passed"].value_counts().sort_index().to_dict()
n = len(df_ranked); passed = int((df_ranked["layers_passed"] >= 3).sum())
print("layers_passed distribution:", dist)
print(f"engagement hit rate (pass all engaged layers): {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")
if "paradigm" in df_ranked.columns:
    for p, g in df_ranked.groupby("paradigm"):
        gp = int((g["layers_passed"] >= 3).sum())
        print(f"  {p:12s}: {gp}/{len(g)} ({100*gp/max(len(g),1):.1f}%)")

## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Filtering applied to the **engaged** subunits (pae = worst of β/γc); per-subunit paes carried through.
- [ ] Survival-at-each-layer reported (funnel figure `results/p13_survival.png`).
- [ ] Honest **engagement** hit-rate accounting (N pass / N generated).
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — the subunit-selectivity profile + stability-vs-native analysis.